# Day 48 — Customer Intelligence Platform: Preprocessing Pipeline

This notebook demonstrates a reproducible preprocessing workflow using **pandas** and **scikit-learn**. It covers validation, missing values, feature engineering, categorical encoding, numerical scaling, train/test separation, and persistence of the fitted transformer.

## 1. Imports and configuration

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT))
from src.preprocessing.pipeline import build_preprocessor, engineer_features

DATA_PATH = ROOT / 'data' / 'raw' / 'customer_data.csv'
df = pd.read_csv(DATA_PATH)
df.head()

## 2. Data quality inspection

In [ ]:
print(df.shape)
display(df.info())
display(df.isna().sum().sort_values(ascending=False))
print('Duplicate rows:', df.duplicated().sum())

## 3. Cleaning and feature engineering

The feature engineering layer creates customer-level behavioral indicators. Missing values are deliberately handled inside the sklearn transformer so the same rules can be reused at inference time.

In [ ]:
df = df.drop_duplicates().copy()
df = engineer_features(df)
df.head()

## 4. Define target and features

For this demonstration, `complaints` is treated as a target-like business outcome. In a real capstone, replace it with the final prediction target selected during the modeling phase.

In [ ]:
target = 'complaints'
X = df.drop(columns=[target, 'customer_id'])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Train:', X_train.shape, 'Test:', X_test.shape)

## 5. Build reusable preprocessing pipeline

In [ ]:
preprocessor, numerical_features, categorical_features = build_preprocessor(X_train)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print('Numerical:', numerical_features)
print('Categorical:', categorical_features)
print('Processed train shape:', X_train_processed.shape)
print('Processed test shape:', X_test_processed.shape)

## 6. Save processed datasets

The transformed matrices are exported as CSV files so the modeling stage can consume a stable representation.

In [ ]:
import numpy as np
from joblib import dump

processed_dir = ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

feature_names = preprocessor.get_feature_names_out()
pd.DataFrame(X_train_processed, columns=feature_names).to_csv(processed_dir / 'X_train.csv', index=False)
pd.DataFrame(X_test_processed, columns=feature_names).to_csv(processed_dir / 'X_test.csv', index=False)
y_train.to_csv(processed_dir / 'y_train.csv', index=False)
y_test.to_csv(processed_dir / 'y_test.csv', index=False)
dump(preprocessor, ROOT / 'models' / 'preprocessor.joblib')

print('Saved processed data and fitted preprocessor.')

## 7. Pipeline principles

- Fit transformations only on training data to reduce data leakage.
- Use `handle_unknown='ignore'` so unseen categories do not break inference.
- Keep feature engineering deterministic and reusable.
- Persist the fitted transformer so future inference uses the same transformations.